# Chapter 12 Companion Notebook: Clustering: Wholesale Customer K-Means

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch12_Clustering_Wholesale_Customer_KMeans.ipynb)

This notebook accompanies Chapter 12 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Wholesale customer: K-means clustering
- https://archive.ics.uci.edu/ml/datasets/wholesale+customers

Use 'Wholesale customer.csv'. 
- customerID: customer ID
- fresh: annual spending on fresh products (continuous)  
- milk: annual spending on milk products (continuous)  
- grocery: annual spending on grocery products (continuous)  
- frozen: annual spending on frozen products (continuous)  
- detergents_paper: annual spending on detergents and paper products (continuous)  
- delicatessen: annual spending on delicatessen products (continuous)  

In [ ]:
import pandas as pd
import numpy as np
from sklearn import cluster
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('Wholesale customer.csv')
df.head()

### Using the variables except CustomerID, apply the elbow method to determine the optimal number of clusters for k-means clustering (k=2~10). How many clusters would you choose?"

In [ ]:
# Run k-means clustering with k=2~10. Plot the errors corresponding to each k.

ss = []
krange = range(2,11)  # 2~10
x=df.iloc[:, 1:]
for i in krange:
    m = cluster.KMeans(n_clusters=i).fit(x)
    error = m.inertia_
    ss.append(error)
    
plt.plot(krange, ss)
plt.xlabel('Number of Clusters')
plt.ylabel('Sum of Squared Errors')

- 5 clusters seems reasonable

### 3. Conduct k-means clustering using the best number of clusters you found. Add predicted cluster labels to the data

In [ ]:
m = cluster.KMeans(n_clusters=5, random_state=10).fit(x)
df['cluster'] = m.labels_  # cluster labels
df.head()

### 3. How many customers are in each cluster?

In [ ]:
# Count the number of customers in each cluster
df['cluster'].value_counts().sort_index()
 # df.groupby('cluster').count()  # alternative

### 4. Which customer segment has the highest average sales in the Grocery category?

In [ ]:
# Calculate the average sales in the Grocery category for each cluster
grocery_avg = df.groupby('cluster')['Grocery'].mean()

# Find the highest and lowest average sales clusters
highest_cluster = grocery_avg.idxmax()
highest_sales = grocery_avg.max()
highest_cluster, highest_sales

- idxmax() returns the index (or label) of the maximum value in a given Series. It finds the cluster with the highest average Grocery sales.

In [ ]:
# Inspect cluster centers

print(m.cluster_centers_)

### 5. Report the top 5 customers whose grocery sales are the highest within the highest grocery sales segment.

In [ ]:
# Filter the customers belonging to the highest grocery sales cluster
top_cluster = df[df['cluster'] == highest_cluster]

# Get the top 3 customers with the highest grocery sales in this cluster
top_cluster.nlargest(5, 'Grocery')

### 8. Increase the number of times that k-means clustering will be run with different initial centroids to 20. What is the lowest SSE (some of squared error)? What is the actual number of runs to reach the solution (i.e., convergence)?

In [ ]:
m = cluster.KMeans(init='k-means++', n_init=20, n_clusters=5).fit(x)
df['cluster'] = m.labels_
df.groupby('cluster').count()

- `init='k-means++'`(default): Selects initial cluster centers in a smart way to speed up convergence
  - e.g.) Select the first center randomly, find the points that are farther to the first center and assign the second cluster center nearby those points
- `init='random'`: Random initialization
- `n_init`: Number of times the algorithm is run with different initial centroid (default=10). Final results will be  the best output in terms of inertia.

In [ ]:
m.inertia_  

- `m.inertia_`: returns the lowest value of the sum of squared distance of samples to cluster center

In [ ]:
m.n_iter_  

- `m.n_iter_`: returns the number of iterations that the algorithm executed until it converged.
    - The solution(convergence) was reached before 20 runs.
    - This number might be different every time you run the algorithm